# Aviation Anomaly Detection — Experiment Launcher

This notebook runs the full benchmark pipeline from Google Colab Pro.

**Setup:** Make sure `aviation_anomaly/` is in your Google Drive with:
- `workflow_python/` — all Python modules
- `data/` — train/test `.npy` files

**Runtime:** Change to GPU via Runtime → Change runtime type → T4 GPU

## 1. Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ──────────────────────────────────────────────────────────────────────
# UPDATE THIS PATH to match your Google Drive folder structure.
# It should point to the root of the aviation_anomaly project.
# ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/Flight-Safety'

os.chdir(os.path.join(PROJECT_ROOT, 'workflow_python'))
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# Verify GPU is available
import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
if not gpus:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Verify data is accessible
data_dir = os.path.join(PROJECT_ROOT, 'data')
print(f'Data directory: {data_dir}')
print(f'Data files: {os.listdir(data_dir)}')

## 2. List Available Models

In [ ]:
!python main.py --list

## 3. Smoke Test (2 epochs, 1 model)

Run this first to make sure everything works before committing to the full benchmark.

In [ ]:
!python main.py --model base_gru --cp 100 --epochs 2 --folds 2

## 4. Full Benchmark

Uncomment the experiment you want to run. Estimated times (Colab Pro T4):
- Single model, single checkpoint: ~5–15 min
- Single model, all checkpoints: ~20–60 min
- All models, all checkpoints: ~3–6 hours
- Full benchmark + explainability: ~8–12 hours

In [ ]:
# ── Option A: Single model, single checkpoint ──
# !python main.py --model base_gru --cp 100 --epochs 30

# ── Option B: Single model, all checkpoints ──
# !python main.py --model gtn --cp all --epochs 30

# ── Option C: All models, all checkpoints (no explainability) ──
# !python main.py --model all --cp all --epochs 30

# ── Option D: Full benchmark with all explainability ──
# !python main.py --model all --cp all --epochs 30 --all-explain

## 5. Run Individual Models

Use these cells to run models one at a time (useful if Colab disconnects).

In [ ]:
# Tier 1: Aviation Baselines
!python main.py --model base_gru --cp all --epochs 30 --mc-dropout --gradcam --lime

In [ ]:
!python main.py --model mhcnn_rnn --cp all --epochs 30 --mc-dropout --gradcam --lime

In [ ]:
# Tier 2: Cross-Domain SOTA
!python main.py --model attn_block1 --cp all --epochs 30 --mc-dropout --gradcam --lime

In [ ]:
!python main.py --model attn_block2 --cp all --epochs 30 --mc-dropout --gradcam --lime

In [ ]:
!python main.py --model gtn --cp all --epochs 30 --mc-dropout --gradcam --lime

In [ ]:
!python main.py --model inception_time --cp all --epochs 30 --mc-dropout --gradcam --lime

In [ ]:
# Tier 3: Early Classification
!python main.py --model earliest --cp all --epochs 30 --mc-dropout --gradcam --lime

## 6. Check Results

In [ ]:
import json
import os

results_dir = os.path.join(PROJECT_ROOT, 'results')
if os.path.exists(results_dir):
    result_files = sorted([f for f in os.listdir(results_dir) if f.endswith('.json')])
    print(f'Results saved: {len(result_files)} files')
    for f in result_files:
        with open(os.path.join(results_dir, f)) as fh:
            r = json.load(fh)
        avg = r['cross_validation']['averages']
        print(f"  {f:<35} Acc={avg['test']['accuracy_mean']:.4f}  "
              f"F1={avg['test']['f1_mean']:.4f}  "
              f"AUC={avg['test']['auc_roc_mean']:.4f}")
else:
    print('No results yet — run an experiment first!')